In [21]:
import pandas as pd
from pyfaidx import Fasta
from Bio import Entrez, SeqIO
from Bio.Seq import Seq

In [2]:
base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_procesada_OR_709.csv")

In [3]:
base

,Chr,gene_symbol,SNPs_symbol,SNP_position,effect_allele,alternate_allele,joint_phase_OR,Chr_corr,Gene_symbol_corr,SNP_position_corr,Effect_allele_corr,Alternate_allele_corr,Cadena,Valida_alelos
0,6,GPR126,rs757765789,142758601,T,G,1.06,6,ADGRG6,142758601,T,['G'],+,Coincide perfecto
1,12,SLC2A13,rs1994090,40428561,G,T,12.05,12,SLC2A13,40428561,G,"['A', 'T', 'C']",-,Coincide perfecto
2,12,SLC2A13,rs2708453,40478652,G,T,12.05,12,SLC2A13,40478652,G,"['A', 'T']",-,Coincide perfecto
3,12,SLC2A13,rs4768212,40474147,C,T,12.05,12,SLC2A13,40474147,C,"['A', 'T']",-,Coincide perfecto
4,14,SLC2A15,rs7304281,40465942,T,C,12.05,12,LINC02347,126945694,T,"['G', 'C']",+,Coincide perfecto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
704,17,BRIP1,rs61169879,59917366,T,C,1.09,17,BRIP1,59917366,C,"['A', 'T']",-,Coincide perfecto
705,17,DNAH17,rs666463,76425480,A,T,1.08,17,DNAH17,76425480,A,['T'],-,Coincide perfecto
706,18,ASXL3,rs1941685,31304318,T,G,1.05,18,ASXL3,31304318,G,"['T', 'C']",+,Coincide perfecto
707,20,CRLS1,rs77351827,6006041,T,C,1.08,20,CRLS1,6006041,C,['T'],+,Coincide perfecto


In [23]:
archivo_genoma = "/home/jbs1009/TFM/datosGene4PD/Homo_sapiens.GRCh37.completo.fa"

genoma = SeqIO.index(archivo_genoma, "fasta")

In [ ]:
def extrae_region_sana_OR_aumentado(df_final_fila, genoma, ventana = 50, desplazamiento = 2):
    """
    Permite la extracción de la secuencia sana sintética correspondiente a las regiones flanqueantes en torno a la posición 
    de SNP, teniendo en cuenta el valor de la Odds Ratio para el effect_allele (<1: protector, >1: riesgo). Además, extrae también las
    secuencias sanas equivalente a desplazar la región hacia atrás y hacia adelante tantas posiciones como el valor de 'desplazamiento'

    Parámetros:
    -----------------------
    - df_final_fila (pandas.core.series.Series): instancia del dataframe preprocesado antes de crear el dataset sintético
    - genoma: archivo del genoma de referieca sobre el que se extraen las regiones flanqueantes
    - ventana (int, por defecto 500): tamaño de cada región flanqueante en torno a la posición de SNP
    - desplazamiento (int, por defecto 2): número de posiciones que se desplaza la región resultante hacia atrás y hacia adelante respecto
                                            a la posición inicial central del SNP

    Returns:
    -----------------------
    - secuencias_generadas (list): lista de secuencias de regiones flanqueantes sanas aumentadas con desplazamiento
    """

    cromosoma = str(df_final_fila["Chr_corr"])
    pos_snp = df_final_fila["SNP_position_corr"]

    if float(df_final_fila["joint_phase_OR"]) <= 1:
        alelo_sano = df_final_fila["effect_allele"]

    else:
        alelo_sano = df_final_fila["alternate_allele"]

    secuencia_chr = genoma[cromosoma].seq

    id_snp_python = pos_snp - 1

    cadena = df_final_fila.get("Cadena")

    secuencias_generadas = []

    for desp in range(-desplazamiento, desplazamiento + 1):

        inicio = max(0, id_snp_python - ventana + desp)
        fin = id_snp_python + ventana + 1 + desp

        molde_forward = str(secuencia_chr[inicio : fin]).lower()

        pos_relativa_snp = id_snp_python - inicio

        reg_sana_forward = molde_forward[:pos_relativa_snp] + alelo_sano + molde_forward[pos_relativa_snp + 1:]
    
    
        if cadena == 1 or cadena == "+":
            
            reg_sana = reg_sana_forward

        else:

            reg_sana = str(Seq(reg_sana_forward).reverse_complement())

        secuencias_generadas.append({"Secuencia": reg_sana.lower(), "Desplazamiento": desp, "rsID": df_final_fila["SNPs_symbol"]})

    return secuencias_generadas

In [ ]:
def extrae_region_parkinson_OR_aumentado(df_final_fila, lista_sanas, ventana = 50):
    """
    Permite la extracción de la secuencia sana sintética correspondiente a las regiones flanqueantes en torno a la posición 
    de SNP, teniendo en cuenta el valor de la Odds Ratio para el effect_allele (<1: protector, >1: riesgo). Además, extrae también las
    secuencias sanas equivalente a desplazar la región hacia atrás y hacia adelante.

    Parámetros:
    -----------------------
    - df_final_fila (pandas.core.series.Series): instancia del dataframe preprocesado antes de crear el dataset sintético
    - lista_sanas (list): lista de secuencias de regiones flanqueantes sanas aumentadas con desplazamiento
    - ventana (int, por defecto 500): tamaño de cada región flanqueante en torno a la posición de SNP

    Returns:
    -----------------------
    - secuencias_park_generadas (list): lista de secuencias de regiones flanqueantes de riesgo aumentadas generadas a partir de las secuencias
                                        sanas aumentadas generadas previamente
    """
    
    cadena = df_final_fila.get("Cadena")

    if float(df_final_fila["joint_phase_OR"]) <= 1:

        alelo_parkinson = df_final_fila["alternate_allele"]

    else:
        
        alelo_parkinson = df_final_fila["effect_allele"]

    secuencias_park_generadas = []

    for sana in lista_sanas:

        reg_sana = sana["Secuencia"]
        desp = sana["Desplazamiento"]

        pos_relativa_snp = ventana - desp

        if cadena == 1 or cadena == "+":

            snp = pos_relativa_snp
            nuc = alelo_parkinson
        
        else:

            snp = len(reg_sana) - 1 - pos_relativa_snp
            nuc = str(Seq(alelo_parkinson).complement())

        reg_park = reg_sana[:snp] + nuc.lower() + reg_sana[snp + 1:]

        secuencias_park_generadas.append({"Secuencia": reg_park, "Desplazamiento": desp, "rsID": sana["rsID"]})

    return secuencias_park_generadas

In [7]:
df_flancos50_3109 = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_50_OR_3109_con_rsid_extendida.csv")

In [13]:
df_flancos50_3109

,Secuencia,labels,rsID
0,tatattttcttttgtcacaggcttatttatattcatcttccactgt...,Sano,rs757765789
1,tatattttcttttgtcacaggcttatttatattcatcttccactgt...,Riesgo_PD,rs757765789
2,agtgttttcaaagaatctattgattctaattgctaaccctttttat...,Sano,rs1994090
3,agtgttttcaaagaatctattgattctaattgctaaccctttttat...,Riesgo_PD,rs1994090
4,caacccttatcctgaagctgcctaggggctaccagtcatctgtcaa...,Sano,rs2708453
...,...,...,...
3104,acagaagaagcagatgtcaccgtggggccactgatcttcctggaca...,Sano,rs200481427
3105,aggcatgagccaccatgcccagctgactgtgtctttcagagaactg...,Sano,rs74676082
3106,caccgtggggccactgatcttcctggacaggaggggtgaccatgaa...,Sano,rs143649134
3107,ctggtctccatggacacagaagatgtggtcaggtttgaggttggac...,Sano,rs142370942


In [24]:
todas = []

for i, fila in base.iterrows():

    sanas_aumentadas = extrae_region_sana_OR_aumentado(fila, genoma, ventana = 50, desplazamiento = 2)

    for sana in sanas_aumentadas:

        sana["labels"] = "Sano"
        todas.append(sana)

    riesgo_aumentadas = extrae_region_parkinson_OR_aumentado(fila, sanas_aumentadas, ventana = 50)

    for riesgo in riesgo_aumentadas:

        riesgo["labels"] = "Riesgo_PD"
        todas.append(riesgo)

base_extendida = pd.DataFrame(todas)


In [25]:
base_extendida

,Secuencia,Desplazamiento,rsID,labels
0,tttatattttcttttgtcacaggcttatttatattcatcttccact...,-2,rs757765789,Sano
1,ttatattttcttttgtcacaggcttatttatattcatcttccactg...,-1,rs757765789,Sano
2,tatattttcttttgtcacaggcttatttatattcatcttccactgt...,0,rs757765789,Sano
3,atattttcttttgtcacaggcttatttatattcatcttccactgtg...,1,rs757765789,Sano
4,tattttcttttgtcacaggcttatttatattcatcttccactgtgc...,2,rs757765789,Sano
...,...,...,...,...
7085,atagctgtttttcattaatcgttatatttttataggtttaaaaact...,-2,rs2248244,Riesgo_PD
7086,tagctgtttttcattaatcgttatatttttataggtttaaaaactg...,-1,rs2248244,Riesgo_PD
7087,agctgtttttcattaatcgttatatttttataggtttaaaaactgg...,0,rs2248244,Riesgo_PD
7088,gctgtttttcattaatcgttatatttttataggtttaaaaactggg...,1,rs2248244,Riesgo_PD


In [31]:
base_extendida.to_csv("/home/jbs1009/TFM/datosGene4PD/extras_desp2_flancos50.csv", index = False)

In [32]:
b = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/extras_desp2_flancos50.csv")

In [34]:
b

,Secuencia,Desplazamiento,rsID,labels
0,tttatattttcttttgtcacaggcttatttatattcatcttccact...,-2,rs757765789,Sano
1,ttatattttcttttgtcacaggcttatttatattcatcttccactg...,-1,rs757765789,Sano
2,tatattttcttttgtcacaggcttatttatattcatcttccactgt...,0,rs757765789,Sano
3,atattttcttttgtcacaggcttatttatattcatcttccactgtg...,1,rs757765789,Sano
4,tattttcttttgtcacaggcttatttatattcatcttccactgtgc...,2,rs757765789,Sano
...,...,...,...,...
7085,atagctgtttttcattaatcgttatatttttataggtttaaaaact...,-2,rs2248244,Riesgo_PD
7086,tagctgtttttcattaatcgttatatttttataggtttaaaaactg...,-1,rs2248244,Riesgo_PD
7087,agctgtttttcattaatcgttatatttttataggtttaaaaactgg...,0,rs2248244,Riesgo_PD
7088,gctgtttttcattaatcgttatatttttataggtttaaaaactggg...,1,rs2248244,Riesgo_PD


In [35]:
b = b[["Secuencia", "labels", "rsID"]]

In [36]:
b

,Secuencia,labels,rsID
0,tttatattttcttttgtcacaggcttatttatattcatcttccact...,Sano,rs757765789
1,ttatattttcttttgtcacaggcttatttatattcatcttccactg...,Sano,rs757765789
2,tatattttcttttgtcacaggcttatttatattcatcttccactgt...,Sano,rs757765789
3,atattttcttttgtcacaggcttatttatattcatcttccactgtg...,Sano,rs757765789
4,tattttcttttgtcacaggcttatttatattcatcttccactgtgc...,Sano,rs757765789
...,...,...,...
7085,atagctgtttttcattaatcgttatatttttataggtttaaaaact...,Riesgo_PD,rs2248244
7086,tagctgtttttcattaatcgttatatttttataggtttaaaaactg...,Riesgo_PD,rs2248244
7087,agctgtttttcattaatcgttatatttttataggtttaaaaactgg...,Riesgo_PD,rs2248244
7088,gctgtttttcattaatcgttatatttttataggtttaaaaactggg...,Riesgo_PD,rs2248244


In [44]:
df_flancos50_aumentada = pd.concat([df_flancos50_3109, b], ignore_index = True)

In [45]:
df_flancos50_aumentada

,Secuencia,labels,rsID
0,tatattttcttttgtcacaggcttatttatattcatcttccactgt...,Sano,rs757765789
1,tatattttcttttgtcacaggcttatttatattcatcttccactgt...,Riesgo_PD,rs757765789
2,agtgttttcaaagaatctattgattctaattgctaaccctttttat...,Sano,rs1994090
3,agtgttttcaaagaatctattgattctaattgctaaccctttttat...,Riesgo_PD,rs1994090
4,caacccttatcctgaagctgcctaggggctaccagtcatctgtcaa...,Sano,rs2708453
...,...,...,...
10194,atagctgtttttcattaatcgttatatttttataggtttaaaaact...,Riesgo_PD,rs2248244
10195,tagctgtttttcattaatcgttatatttttataggtttaaaaactg...,Riesgo_PD,rs2248244
10196,agctgtttttcattaatcgttatatttttataggtttaaaaactgg...,Riesgo_PD,rs2248244
10197,gctgtttttcattaatcgttatatttttataggtttaaaaactggg...,Riesgo_PD,rs2248244


In [46]:
df_flancos50_aumentada["labels"].value_counts()

labels
Sano         5945
Riesgo_PD    4254
Name: count, dtype: int64

In [47]:
b["labels"].value_counts()

labels
Sano         3545
Riesgo_PD    3545
Name: count, dtype: int64

In [48]:
df_flancos50_aumentada_sin_dups = df_flancos50_aumentada.drop_duplicates(subset = ["Secuencia"])

In [49]:
df_flancos50_aumentada_sin_dups

,Secuencia,labels,rsID
0,tatattttcttttgtcacaggcttatttatattcatcttccactgt...,Sano,rs757765789
1,tatattttcttttgtcacaggcttatttatattcatcttccactgt...,Riesgo_PD,rs757765789
2,agtgttttcaaagaatctattgattctaattgctaaccctttttat...,Sano,rs1994090
3,agtgttttcaaagaatctattgattctaattgctaaccctttttat...,Riesgo_PD,rs1994090
4,caacccttatcctgaagctgcctaggggctaccagtcatctgtcaa...,Sano,rs2708453
...,...,...,...
10193,ctgtttttcattaatcgttatatttttataggtttaaaaactgggc...,Sano,rs2248244
10194,atagctgtttttcattaatcgttatatttttataggtttaaaaact...,Riesgo_PD,rs2248244
10195,tagctgtttttcattaatcgttatatttttataggtttaaaaactg...,Riesgo_PD,rs2248244
10197,gctgtttttcattaatcgttatatttttataggtttaaaaactggg...,Riesgo_PD,rs2248244


In [50]:
df_flancos50_aumentada_sin_dups["labels"].value_counts()

labels
Sano         4704
Riesgo_PD    3015
Name: count, dtype: int64

In [51]:
df_sanas = df_flancos50_aumentada_sin_dups[df_flancos50_aumentada_sin_dups["labels"] == "Sano"]
df_riesgo = df_flancos50_aumentada_sin_dups[df_flancos50_aumentada_sin_dups["labels"] == "Riesgo_PD"]

In [55]:
df_sanas_downsampling = df_sanas.sample(n = len(df_riesgo), random_state = 2026)

In [57]:
base_final_aumentada = pd.concat([df_riesgo, df_sanas_downsampling]).sample(frac = 1, random_state = 2026).reset_index(drop = True)

In [58]:
base_final_aumentada

,Secuencia,labels,rsID
0,ttttcagggaggtagtcctggattactttcagttaattggcccttg...,Riesgo_PD,rs4697508
1,atctaaattattaaagatactggcataacattatttataacattcc...,Riesgo_PD,rs71628662
2,tgcagctccagcctgggtgacagaatgacctgtctcaaaaaaaaaa...,Sano,rs2361113
3,aaaaattgccatttttacaatattaggttacgataggttacaatta...,Riesgo_PD,rs1362858
4,gataatggcttacaagttaatctcctcttgctccctgttacacaca...,Riesgo_PD,rs2736990
...,...,...,...
6025,tctgaaaatgtgacctttgtgctgagaccggaatgacaacaaggag...,Sano,rs9876540
6026,atcatgggggtaatgacttaagcggtggctggcaggaagtacctgt...,Riesgo_PD,rs2553427
6027,cctggccactccaagcatatgtcagtaagtctgtgcccttttattt...,Sano,rs6783485
6028,gcctcaagcagcatcattgcaacagataatgtgttattcacaccca...,Riesgo_PD,rs13243961


In [59]:
base_final_aumentada["labels"].value_counts()

labels
Riesgo_PD    3015
Sano         3015
Name: count, dtype: int64

In [60]:
base_final_aumentada.to_csv("/home/jbs1009/TFM/datosGene4PD/base_aumentada_flancos50_6030.csv", index = False)

In [61]:
base_final_aumentada = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_aumentada_flancos50_6030.csv")

In [62]:
base_final_aumentada

,Secuencia,labels,rsID
0,ttttcagggaggtagtcctggattactttcagttaattggcccttg...,Riesgo_PD,rs4697508
1,atctaaattattaaagatactggcataacattatttataacattcc...,Riesgo_PD,rs71628662
2,tgcagctccagcctgggtgacagaatgacctgtctcaaaaaaaaaa...,Sano,rs2361113
3,aaaaattgccatttttacaatattaggttacgataggttacaatta...,Riesgo_PD,rs1362858
4,gataatggcttacaagttaatctcctcttgctccctgttacacaca...,Riesgo_PD,rs2736990
...,...,...,...
6025,tctgaaaatgtgacctttgtgctgagaccggaatgacaacaaggag...,Sano,rs9876540
6026,atcatgggggtaatgacttaagcggtggctggcaggaagtacctgt...,Riesgo_PD,rs2553427
6027,cctggccactccaagcatatgtcagtaagtctgtgcccttttattt...,Sano,rs6783485
6028,gcctcaagcagcatcattgcaacagataatgtgttattcacaccca...,Riesgo_PD,rs13243961
